In [ ]:
import sys, os
sys.path.append(os.path.abspath("..")) # Allows us to import from grips and qokit directories
import matplotlib.pyplot as plt, networkx as nx, qokit.maxcut as mc, numpy as np
from grips import QAOA_proxy, QAOA_proxy_expectation
from grips import get_simulator, get_expectation, get_result
from grips import plot_heat_map
from grips import TriangleProxy, PaperProxy, NormalProxy

# The following imports and seval statements make Julia proxy functions available
from juliacall import Main as jl
jl.seval('''
using Pkg
Pkg.activate(joinpath(@__DIR__, "../julia"))
Pkg.instantiate()
using JuliaQAOA
''')

/home/spencer/Research/GRIPS/QOKit/qokit/qaoa_circuit_portfolio.py:29: SyntaxWarning: invalid escape sequence '\s'
  """
/home/spencer/Research/GRIPS/QOKit/qokit/qaoa_circuit_portfolio.py:56: SyntaxWarning: invalid escape sequence '\s'
  """
/home/spencer/Research/GRIPS/QOKit/qokit/portfolio_optimization.py:26: SyntaxWarning: invalid escape sequence '\s'
  """
/home/spencer/Research/GRIPS/QOKit/qokit/portfolio_optimization.py:170: SyntaxWarning: invalid escape sequence '\s'
  """
/home/spencer/Research/GRIPS/QOKit/qokit/labs.py:155: SyntaxWarning: invalid escape sequence '\i'
  """Compute LABS energy values from a string of spins
/home/spencer/Research/GRIPS/QOKit/qokit/labs.py:182: SyntaxWarning: invalid escape sequence '\i'
  """Convenience function
/home/spencer/Research/GRIPS/QOKit/qokit/labs.py:203: SyntaxWarning: invalid escape sequence '\i'
  """Compute energy values from a string of spins
/home/spencer/Research/GRIPS/QOKit/qokit/labs.py:234: SyntaxWarning: invalid escape sequen

[juliapkg] Found dependencies: /home/spencer/Research/GRIPS/qokitvenv/lib/python3.12/site-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /home/spencer/Research/GRIPS/qokitvenv/lib/python3.12/site-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia ~1.9, ^1.10.3
[juliapkg] Using Julia 1.11.6 at /home/spencer/.julia/juliaup/julia-1.11.6+0.x64.linux.gnu/bin/julia
[juliapkg] Using Julia project at /home/spencer/Research/GRIPS/qokitvenv/julia_env
[juliapkg] Writing Project.toml:
             [deps]
             PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
             OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
             [compat]
             PythonCall = "=0.9.27"
             OpenSSL_jll = "~3.0"
[juliapkg] Installing packages:
             import Pkg
             Pkg.Registry.update()
             Pkg.add([
               Pkg.PackageSpec(name="PythonCall", uuid="6099a3de-0909-46bc-b1f4-468b9a2dfc0d"),
               Pkg.PackageSpec(name="OpenSSL_

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed TableTraits ───────────────── v1.0.1
   Installed MicroMamba ────────────────── v0.1.14
   Installed OpenSSL_jll ───────────────── v3.0.16+0
   Installed micromamba_jll ────────────── v1.5.12+0
   Installed Tables ────────────────────── v1.12.1
   Installed Pidfile ───────────────────── v1.3.0
   Installed JSON3 ─────────────────────── v1.14.3
   Installed PythonCall ────────────────── v0.9.27
   Installed IteratorInterfaceExtensions ─ v1.0.0
   Installed DataValueInterfaces ───────── v1.0.0
   Installed UnsafePointers ────────────── v1.0.0
   Installed pixi_jll ──────────────────── v0.41.3+0
   Installed CondaPkg ──────────────────── v0.2.30
   Installed StructTypes ───────────────── v1.11.0
    Updating `~/Research/GRIPS/qokitvenv/julia_env/Project.toml`
  [6099a3de] + PythonCall v0.9.27
⌅ [458c3c95] + OpenSSL_jll v3.0.16+0
    Updating `~/Research/GRIPS/qokitvenv/julia_env/Manif

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


  Activating project at `~/Research/GRIPS/QOKit/julia`
    Updating registry at `~/.julia/registries/General.toml`
   Installed StatsFuns ─────────────── v1.5.0
   Installed HypergeometricFunctions ─ v0.3.28
   Installed Rmath ─────────────────── v0.8.0
   Installed QuadGK ────────────────── v2.11.2
   Installed FillArrays ────────────── v1.13.0
   Installed SpecialFunctions ──────── v2.5.1
   Installed PDMats ────────────────── v0.11.35
   Installed StaticArrays ──────────── v1.9.14
   Installed Rmath_jll ─────────────── v0.5.1+0
   Installed OpenSpecFun_jll ───────── v0.5.6+0
   Installed DataStructures ────────── v0.19.1
   Installed Distributions ─────────── v0.25.120
    Updating `~/Research/GRIPS/QOKit/julia/Project.toml`
  [31c24e10] + Distributions v0.25.120
  [6099a3de] + PythonCall v0.9.27
  [90137ffa] + StaticArrays v1.9.14
    Updating `~/Research/GRIPS/QOKit/julia/Manifest.toml`
  [66dad0bd] + AliasTables v1.1.3
  [992eb4ea] + CondaPkg v0.2.30
  [9a962f9c] + DataAPI v1.16.

In [ ]:

# Define parameter ranges (QOKit convention: phase gate is exp(-iγC/2))
gammas = np.linspace(0, 2 * np.pi, 50)
betas = np.linspace(0, np.pi, 50)
# Probabilities for the Erdos-Renyi graph generation
probabilities = [0.5]
# Whether to use python or julia implementations of the proxies
use_julia = True


def collect_QAOA_parameter_data(G: nx.Graph, gammas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    expectations = np.zeros([len(gammas), len(betas)])
    N = G.number_of_nodes()
    ising_model = mc.get_maxcut_terms(G)
    sim = get_simulator(N, ising_model)

    for i in range(len(gammas)):
        for j in range(len(betas)):
            gamma = np.array([gammas[i]])
            beta = np.array([betas[j]])
            result = get_result(N, ising_model, gamma, beta, sim)
            expectations[i][j] = get_expectation(N, ising_model, gamma, beta, sim, result)

    return expectations


def collect_proxy_parameter_data(proxy, gammas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    expectations = np.zeros([len(gammas), len(betas)])
    for i in range(len(gammas)):
        for j in range(len(betas)):
            gamma = np.array([gammas[i]])
            beta = np.array([betas[j]])
            proxy_amplitudes = QAOA_proxy(proxy, gamma, beta)
            final_proxy_amplitudes = proxy_amplitudes[-1]
            expectation = QAOA_proxy_expectation(proxy, final_proxy_amplitudes)
            expectations[i][j] = expectation

    return expectations


for p in probabilities:
    for N in range(2, 7):
        G = nx.erdos_renyi_graph(N, p, seed=18) # generate graphs
        M = G.number_of_edges()

        # Get the proxies
        if use_julia:
            # Julia Version
            triangle_proxy = jl.TriangleProxy(M, N)
            paper_proxy = jl.PaperProxy(M, N, p)
            normal_proxy = jl.NormalProxy(M, N, 1, 1, 1) # Bad parameters, this proxy is not good like this
        else:
            # Python version
            triangle_proxy = TriangleProxy(M, N) # Will not be good for p != 0.5
            paper_proxy = PaperProxy(M, N, p)
            normal_proxy = NormalProxy(M, N, 1, 1, 1) # Bad parameters, this proxy is not good like this


        # Define data paths
        base_data_path = f"Data/Expectation_Heatmaps/Erdos_Renyi/ER_p={p}"
        os.makedirs(base_data_path, exist_ok=True) # Create directories if they do not exist
        qaoa_data_path = os.path.join(base_data_path, f"QAOA_N={N}_M={M}.npz")
        paper_proxy_data_path = os.path.join(base_data_path, f"PaperProxy_N={N}_M={M}.npz")
        triangle_proxy_data_path = os.path.join(base_data_path, f"TriangleProxy_N={N}_M={M}.npz")
        normal_proxy_data_path = os.path.join(base_data_path, f"NormalProxy_N={N}_M={M}.npz")

        # Define image save paths
        base_img_path = f"Plots/Expectation_Heatmaps/Erdos_Renyi/ER_p={p}"
        os.makedirs(base_img_path, exist_ok=True) # Create directories if they do not exist
        qaoa_img_path = os.path.join(base_img_path, f"QAOA_N={N}_M={M}.png")
        paper_proxy_img_path = os.path.join(base_img_path, f"paper_proxy_N={N}_M={M}.png")
        triangle_proxy_img_path = os.path.join(base_img_path, f"triangle_proxy_N={N}_M={M}.png")
        normal_proxy_img_path = os.path.join(base_img_path, f"normal_proxy_N={N}_M={M}.png")


        # Collect data if it has not already been collected and saved, the Plot
        if not os.path.exists(qaoa_data_path):
            qaoa_expectations = collect_QAOA_parameter_data(G, gammas, betas)
            np.savez(qaoa_data_path, expectations=qaoa_expectations, gammas=gammas, betas=betas)

        # QAOA Plot
        data = np.load(qaoa_data_path)
        qaoa_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, qaoa_expectations, f"True QAOA Expectation (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(qaoa_img_path)

        if not os.path.exists(paper_proxy_data_path):
            paper_proxy_expectations = collect_proxy_parameter_data(paper_proxy, gammas, betas)
            np.savez(paper_proxy_data_path, expectations=paper_proxy_expectations, gammas=gammas, betas=betas)

        # Paper Proxy Plot
        data = np.load(paper_proxy_data_path)
        paper_proxy_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, paper_proxy_expectations, f"Expectation Proxies from Paper (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(paper_proxy_img_path)
        plt.show()

        if not os.path.exists(triangle_proxy_data_path):
            triangle_proxy_expectations = collect_proxy_parameter_data(triangle_proxy, gammas, betas)
            np.savez(triangle_proxy_data_path, expectations=triangle_proxy_expectations, gammas=gammas, betas=betas)

        # Triangle Proxy Plot
        data = np.load(triangle_proxy_data_path)
        triangle_proxy_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, triangle_proxy_expectations, f"Expectation Proxies from Us (Triangle) (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(triangle_proxy_img_path)
        plt.show()

        if not os.path.exists(normal_proxy_data_path):
            normal_proxy_expectations = collect_proxy_parameter_data(normal_proxy, gammas, betas)
            np.savez(normal_proxy_data_path, expectations=normal_proxy_expectations, gammas=gammas, betas=betas)

        # Normal Proxy Plot
        data = np.load(normal_proxy_data_path)
        normal_proxy_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, normal_proxy_expectations, f"Expectation Proxies from Us (Normal) (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(normal_proxy_img_path)
        plt.show()